In [2]:
!pip install pandas
!pip install numpy

In [3]:
import pandas as pd
import numpy as np

In [4]:
flights = pd.read_csv('raw/flight_data_2018_2024.csv', low_memory=False)
print(flights.shape)
flights.head()

(582425, 120)


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Marketing_Airline_Network,Operated_or_Branded_Code_Share_Partners,DOT_ID_Marketing_Airline,IATA_Code_Marketing_Airline,...,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Duplicate,Unnamed: 119
0,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN
1,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN
2,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN
3,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN
4,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN


In [5]:
# Select and rename only the columns needed for the model
df = flights[['IATA_Code_Marketing_Airline', 'Origin', 'Dest', 'CRSDepTime', 'DayOfWeek', 'Month', 'Distance', 'DepDel15', 'Cancelled']].copy()
df.columns = ['airline', 'origin', 'destination', 'crs_dep_time', 'day_of_week', 'month', 'distance', 'delayed', 'cancelled']

# Drop cancelled flights (no departure delay outcome)
df = df[df['cancelled'] == 0.0].drop(columns='cancelled')

# Convert CRSDepTime (e.g. 1738) to dep_hour (17)
df['dep_hour'] = (df['crs_dep_time'] // 100).astype(int)
df = df.drop(columns='crs_dep_time')

# Drop rows with missing target or key features
df = df.dropna(subset=['delayed', 'airline', 'origin', 'destination', 'distance'])
df['delayed'] = df['delayed'].astype(int)
df['distance'] = df['distance'].astype(int)

print(df.shape)
print(df['delayed'].value_counts())
df.head()

(560352, 8)
delayed
0    430916
1    129436
Name: count, dtype: int64


,airline,origin,destination,day_of_week,month,distance,delayed,dep_hour
0,UA,MHT,EWR,7,1,209,1,17
1,UA,IAD,EWR,7,1,212,0,8
2,UA,EWR,MHT,7,1,209,1,15
3,UA,STL,ORD,7,1,258,0,6
4,UA,STL,IAD,7,1,696,1,13


In [6]:
# Strip whitespace from string columns
for col in ['airline', 'origin', 'destination']:
    df[col] = df[col].str.strip()

print('Unique airlines:', df['airline'].nunique())
print('Unique origins:', df['origin'].nunique())
print('Unique destinations:', df['destination'].nunique())

Unique airlines: 10
Unique origins: 351
Unique destinations: 351


In [7]:
df.to_csv('processed/flights_processed.csv', index=False)
print('Saved to processed/flights_processed.csv')

Saved to processed/flights_processed.csv
